# Ports & Refs

kfnetlist uses three types to identify connection points in a netlist. Each
type is a Rust-backed, hashable, orderable value object.

| Type | Use case |
|------|----------|
| `NetlistPort` | Cell-level port — a top-level pin visible outside the cell |
| `PortRef` | Reference to a port on a named instance |
| `PortArrayRef` | Reference to a port on a specific element of an array instance |

All three are valid **net members** — they can be passed to `Netlist.create_net()`
and stored inside a `Net`.

In [1]:
from kfnetlist import NetlistPort, PortArrayRef, PortRef

## NetlistPort

A `NetlistPort` represents a cell-level port identified solely by its name.

In [2]:
p = NetlistPort(name="in")
print(f"name: {p.name}")
print(f"hash: {hash(p)}")
print(f"equal: {p == NetlistPort(name='in')}")

name: in
hash: 3504002876683576982
equal: True


## PortRef

A `PortRef` identifies a port on a specific instance by instance name and
port name.

In [3]:
ref = PortRef(instance="mmi1", port="o2")
print(f"instance: {ref.instance}, port: {ref.port}")
print(f"hash: {hash(ref)}")

instance: mmi1, port: o2
hash: -3758625386664795898


`as_python_str()` returns a human-readable representation, optionally using
a custom instance variable name.

In [4]:
print(ref.as_python_str())
print(ref.as_python_str("splitter"))

mmi1['o2']
splitter['o2']


## PortArrayRef

A `PortArrayRef` extends `PortRef` with array indices `ia` and `ib` to
identify a port on a specific element of an array instance.

In [5]:
aref = PortArrayRef(instance="pad_array", port="p1", ia=2, ib=0)
print(f"instance: {aref.instance}, port: {aref.port}, ia: {aref.ia}, ib: {aref.ib}")
print(aref.as_python_str())

instance: pad_array, port: p1, ia: 2, ib: 0
pad_array['p1', 2, 0]


## Ordering

The three types have a defined ordering: `NetlistPort < PortRef < PortArrayRef`.
Within each type, ordering is lexicographic by fields. This ordering is used
by `Net.sort()` and `Netlist.sort()` to produce stable, reproducible output.

In [6]:
items = [
    PortRef(instance="b", port="o1"),
    NetlistPort(name="out"),
    PortArrayRef(instance="a", port="p1", ia=0, ib=0),
    NetlistPort(name="in"),
    PortRef(instance="a", port="o1"),
]
for item in sorted(items):
    print(f"  {type(item).__name__}: {item}")

  NetlistPort: NetlistPort(name='in')
  NetlistPort: NetlistPort(name='out')
  PortRef: a['o1']
  PortRef: b['o1']
  PortArrayRef: a['p1', 0, 0]


## Hashing and equality

All three types are hashable and support equality comparison, so they can be
used in sets and as dict keys.

In [7]:
s = {
    NetlistPort(name="in"),
    PortRef(instance="wg1", port="o1"),
    NetlistPort(name="in"),  # duplicate
}
print(f"Set size (with duplicate): {len(s)}")
assert PortRef(instance="a", port="b") == PortRef(instance="a", port="b")
print("Equality ✓")

Set size (with duplicate): 2
Equality ✓


## Serialization

Each type supports `to_json()` / `from_json()` and `to_dict()` / `from_dict()`.

In [8]:
ref = PortRef(instance="mmi1", port="o2")
print("JSON:", ref.to_json())
print("Dict:", ref.to_dict())

ref2 = PortRef.from_json(ref.to_json())
assert ref == ref2
print("Round-trip ✓")

JSON: {"instance":"mmi1","port":"o2"}
Dict: {'instance': 'mmi1', 'port': 'o2'}
Round-trip ✓


## Summary

| Type | Fields | Purpose |
|------|--------|---------|
| `NetlistPort` | `name` | Cell-level port |
| `PortRef` | `instance`, `port` | Port on a named instance |
| `PortArrayRef` | `instance`, `port`, `ia`, `ib` | Port on an array element |

## See Also

| Topic | Where |
|-------|-------|
| Netlist data model | [Netlist Model](netlist_model.py) |
| Serialization details | [Serialization](serialization.py) |